# Lecture 6: What should we match?

Code behind today's slides. There is no separate lab file for this lecture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # set once, here -- never inside a simulator function

# Section 1: Why not keep adding summaries?

## A worked example: adding a third summary

Recall estimating `(beta, gamma)` in the SIR model. Using final size and early growth rate gives an ok estimate of both. What if we add a third summary, `I_2`, the raw case count on day 2?

In [ ]:
def simulate_sir(n_steps, beta, gamma, S0, I0, R0, rng):
    N = S0 + I0 + R0
    S = np.zeros(n_steps + 1, dtype=int); I = np.zeros(n_steps + 1, dtype=int); R = np.zeros(n_steps + 1, dtype=int)
    S[0], I[0], R[0] = S0, I0, R0
    for t in range(n_steps):
        p_infect = 1 - (1 - beta / N) ** I[t]
        delta_I = rng.binomial(S[t], p_infect)
        delta_R = rng.binomial(I[t], gamma)
        S[t + 1] = S[t] - delta_I
        I[t + 1] = I[t] + delta_I - delta_R
        R[t + 1] = R[t] + delta_R
    return {"S": S, "I": I, "R": R}

N_sir, I0_sir, n_steps_sir, window_sir, day2 = 1000, 20, 60, 8, 2
beta0, gamma0 = 0.30, 0.10

def summary_once(beta, gamma, rng):
    sim = simulate_sir(n_steps_sir, beta, gamma, N_sir - I0_sir, I0_sir, 0, rng)
    final_size = N_sir - sim["S"][-1]
    I_early = sim["I"][:window_sir + 1]
    t_early = np.arange(window_sir + 1)
    growth_rate = np.polyfit(t_early, np.log(np.maximum(I_early, 1)), 1)[0]
    return np.array([final_size, growth_rate, sim["I"][day2]])

## Is $I_2$ informative, redundant, or weak?

In [ ]:
reps_truth = np.array([summary_once(beta0, gamma0, rng) for _ in range(500)])
print("correlation matrix:\n", np.corrcoef(reps_truth, rowvar=False))
print("sd:", reps_truth.std(axis=0, ddof=1))

# sensitivity: finite-difference approximation to d E[s]/d beta, evaluated
# at beta0 -- a 10% bump in beta, divided by the size of that bump
m0 = reps_truth.mean(axis=0)
delta_beta = 0.1 * beta0
reps_bump = np.array([summary_once(beta0 + delta_beta, gamma0, rng) for _ in range(500)])
m1 = reps_bump.mean(axis=0)
sensitivity = (m1 - m0) / delta_beta
print("sensitivity-to-noise ratio:", sensitivity / reps_truth.std(axis=0, ddof=1))

$I_2$ is not simply redundant with either of the other two summaries, and its sensitivity-to-noise ratio is lower than growth rate's.

## Including $I_2$ degrades performance

Build a `(beta, gamma)` grid, estimate `Omega_hat^{-1}` from a modest simulation budget (`B = 10`), and compare RMSE using 2 summaries vs. 3.

In [ ]:
beta_grid = np.linspace(0.15, 0.55, 17)
gamma_grid = np.linspace(0.05, 0.20, 17)
grid_beta, grid_gamma = np.meshgrid(beta_grid, gamma_grid, indexing="ij")
grid_beta = grid_beta.ravel(); grid_gamma = grid_gamma.ravel()
B_grid = 80

def summary_avg(beta, gamma, B, rng):
    return np.mean([summary_once(beta, gamma, rng) for _ in range(B)], axis=0)

grid_summaries = np.array([summary_avg(b, g, B_grid, rng) for b, g in zip(grid_beta, grid_gamma)])

In [ ]:
def estimate_bg(obs, Winv, mode):
    if mode == "two":
        g = grid_summaries[:, :2] - obs[:2]
    else:
        g = grid_summaries - obs
    Q = np.einsum("ij,jk,ik->i", g, Winv, g)
    idx = np.argmin(Q)
    return np.array([grid_beta[idx], grid_gamma[idx]])

def run_comparison(B_omega, rng, n_outer=15, n_reps=50):
    rmse2 = np.zeros(n_outer); rmse3 = np.zeros(n_outer)
    for o in range(n_outer):
        reps = np.array([summary_once(beta0, gamma0, rng) for _ in range(B_omega)])
        Winv2 = np.linalg.inv(np.cov(reps[:, :2], rowvar=False))
        Winv3 = np.linalg.inv(np.cov(reps, rowvar=False))
        err2 = np.zeros((n_reps, 2)); err3 = np.zeros((n_reps, 2))
        for i in range(n_reps):
            obs = summary_once(beta0, gamma0, rng)
            e2 = estimate_bg(obs, Winv2, "two")
            e3 = estimate_bg(obs, Winv3, "three")
            err2[i] = e2 - [beta0, gamma0]
            err3[i] = e3 - [beta0, gamma0]
        rmse2[o] = np.sqrt(np.mean(np.sum(err2 ** 2, axis=1)))
        rmse3[o] = np.sqrt(np.mean(np.sum(err3 ** 2, axis=1)))
    return rmse2, rmse3

rmse2, rmse3 = run_comparison(B_omega=10, rng=rng)
print("mean RMSE, 2 summaries:", round(rmse2.mean(), 4))
print("mean RMSE, 3 summaries:", round(rmse3.mean(), 4))
print("fraction of trials where 3 summaries is worse:", np.mean(rmse3 > rmse2))

A genuinely valid, weak-but-not-redundant summary made the estimate worse on average, and worse in most individual trials -- even though we used the textbook-correct weighting rule, not $W=I$.

# Section 2: MMD as histogram distance

Running example for this section and the next: many independent households, each starting with one index case. The recorded datum per household is the total number infected, including the index case -- an outcome in `{1,...,K}`.

In [ ]:
def household_sir(n_steps, beta, gamma, S0, I0, R0, rng):
    return simulate_sir(n_steps, beta, gamma, S0, I0, R0, rng)

def total_infected(K, beta, gamma, rng, n_steps=60):
    sim = household_sir(n_steps, beta, gamma, K - 1, 1, 0, rng)
    return K - sim["S"][-1]

K = 8
gamma0 = 0.3
beta_obs = 2.5 * gamma0   # "observed": R0 = 2.5
beta_cand = 1.5 * gamma0  # a candidate theta: R0 = 1.5

## Comparing observed and simulated data using histograms

In [ ]:
n_obs = 300
obs_counts = np.array([total_infected(K, beta_obs, gamma0, rng) for _ in range(n_obs)])
sim_counts = np.array([total_infected(K, beta_cand, gamma0, rng) for _ in range(n_obs)])
p_hat = np.array([(obs_counts == j).mean() for j in range(1, K + 1)])
q_hat = np.array([(sim_counts == j).mean() for j in range(1, K + 1)])
print("p_hat:", p_hat)
print("q_hat:", q_hat)
print("d^2:", np.sum((p_hat - q_hat) ** 2))  # the simplest histogram distance

For comparison, two independent draws of 300 households at the same $\theta$ as "observed" give a much smaller $d^2$ -- pure sampling noise:

In [ ]:
sim_counts_same_theta = np.array([total_infected(K, beta_obs, gamma0, rng) for _ in range(n_obs)])
q_hat_same = np.array([(sim_counts_same_theta == j).mean() for j in range(1, K + 1)])
print("d^2 (same theta):", np.sum((p_hat - q_hat_same) ** 2))

In [ ]:
categories = np.arange(1, K + 1)
fig, ax = plt.subplots()
width = 0.35
ax.bar(categories - width / 2, p_hat, width, label="observed (p-hat)")
ax.bar(categories + width / 2, q_hat, width, label="candidate theta (q-hat)")
ax.set_xlabel("total infected per household (1-8 possible)")
ax.legend()
plt.show()

## Checking the general MMD definition against the histogram case

The general empirical definition of MMD, for any kernel `k` and samples `x_1,...,x_n ~ P`, `y_1,...,y_m ~ Q`: `MMD^2 = mean_ii'(k(x_i,x_i')) + mean_jj'(k(y_j,y_j')) - 2 mean_ij(k(x_i,y_j))`. For the equality kernel, this reduces algebraically to `sum_j (p_hat_j - q_hat_j)^2`:

In [ ]:
Kxx_avg = np.mean(obs_counts[:, None] == obs_counts[None, :])
Kyy_avg = np.mean(sim_counts[:, None] == sim_counts[None, :])
Kxy_avg = np.mean(obs_counts[:, None] == sim_counts[None, :])
general_form = Kxx_avg + Kyy_avg - 2 * Kxy_avg
direct_form = np.sum((p_hat - q_hat) ** 2)
print("general:", general_form, " direct:", direct_form)

In [ ]:
outcomes = np.arange(1, K + 1)
K_equality = (outcomes[:, None] == outcomes[None, :]).astype(float)
print(K_equality)
print((p_hat - q_hat) @ K_equality @ (p_hat - q_hat))  # matches sum((p-q)^2) above

## SMM on the category frequencies

Define one summary per possible value of `X` -- its sample proportion, `s_j(X_1,...,X_n) = p_hat_j` for `j = 1,...,8` -- and run ordinary SMM with this eight-dimensional summary vector and `W = I`:

In [ ]:
g_theta = p_hat - q_hat
W_identity = np.eye(len(g_theta))
Q_smm = g_theta @ W_identity @ g_theta  # W = I
print(Q_smm)
print(np.sum((p_hat - q_hat) ** 2))  # identical to the discrete MMD objective above

$Q(\theta) = g(\theta)^\top W g(\theta)$ with $W=I$ is exactly MMD between observed and simulated data.

# Section 3: Smooth kernels

Same total-infected support, 1 through 8 -- these are ordered counts, so a household with 2 people infected should plausibly count as more similar to one with 3 than to one with 8. We use the Gaussian kernel as the less-stringent alternative to the equality kernel.

In [ ]:
h_k = 1.5
K_smooth = np.exp(-(outcomes[:, None] - outcomes[None, :]) ** 2 / (2 * h_k ** 2))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(K_equality, vmin=0, vmax=1, origin="lower",
                      extent=[0.5, K + 0.5, 0.5, K + 0.5])
axes[0].set_title("equality kernel")
im1 = axes[1].imshow(K_smooth, vmin=0, vmax=1, origin="lower",
                      extent=[0.5, K + 0.5, 0.5, K + 0.5])
axes[1].set_title(f"smooth kernel (h={h_k})")
fig.colorbar(im1, ax=axes, shrink=0.8)
plt.show()

The same formula, a richer kernel: for a finite set of outcomes, any kernel can be written as a similarity matrix `K`, and `MMD^2(P,Q) = (p - q)' K (p - q)`. With `K = I` (the equality kernel), this collapses back to `sum_j (p_j - q_j)^2`; with a smooth `K`, a mismatch that moves mass to a neighboring category is weighted differently than one that moves it far away.

In [ ]:
K_smooth_quad = np.exp(-(outcomes[:, None] - outcomes[None, :]) ** 2 / (2 * 1.0 ** 2))
print((p_hat - q_hat) @ K_smooth_quad @ (p_hat - q_hat))

# Section 5: A distribution mean and variance can't distinguish

In [ ]:
def simulate_shape(n, a, rng):
    S = rng.choice([-1, 1], size=n)
    eps = rng.normal(0, np.sqrt(1 - a ** 2), size=n)
    return a * S + eps

# confirm mean/variance invariance
for a in [0, 0.3, 0.6, 0.9]:
    y = simulate_shape(200000, a, rng)
    print(f"a = {a}  mean = {y.mean():.3f}  var = {y.var(ddof=1):.3f}")

In [ ]:
a_show = [0, 0.3, 0.6, 0.9]
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for ax, a in zip(axes, a_show):
    y = simulate_shape(4000, a, rng)
    ax.hist(y, bins=50, color="slateblue", edgecolor="white")
    ax.set_xlim(-3, 3)
    ax.set_title(f"a = {a}")
fig.suptitle("Same mean (0) and variance (1), increasingly separated shape")
plt.show()

## Two different methods for inferring $a$

`a0` is the (unknown, to the estimator) true value; both objectives are built from one fixed observed sample and then evaluated by simulating at each candidate `a`.

In [ ]:
def mmd2_gaussian(x, y, h):
    Kxx = np.exp(-(x[:, None] - x[None, :]) ** 2 / (2 * h ** 2))
    Kyy = np.exp(-(y[:, None] - y[None, :]) ** 2 / (2 * h ** 2))
    Kxy = np.exp(-(x[:, None] - y[None, :]) ** 2 / (2 * h ** 2))
    return Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()

In [ ]:
n_shape = 500
a0 = 0.7
x_obs = simulate_shape(n_shape, a0, rng)
a_grid = np.arange(0, 0.951, 0.025)
B_rep = 30
h_good = 0.3

mmd_vals = np.array([
    np.mean([mmd2_gaussian(x_obs, simulate_shape(n_shape, a, rng), h_good) for _ in range(B_rep)])
    for a in a_grid
])
obs_mean, obs_var = x_obs.mean(), x_obs.var(ddof=1)
smm_vals = np.array([
    np.mean([(obs_mean - (y := simulate_shape(n_shape, a, rng)).mean()) ** 2
             + (obs_var - y.var(ddof=1)) ** 2 for _ in range(B_rep)])
    for a in a_grid
])

print("MMD argmin (should recover a0 = 0.7):", a_grid[np.argmin(mmd_vals)])
print("SMM argmin (should look essentially arbitrary):", a_grid[np.argmin(smm_vals)])

In [ ]:
fig, ax = plt.subplots()
ax.plot(a_grid, smm_vals / smm_vals.max(), "o-", label="SMM: mean + variance")
ax.plot(a_grid, mmd_vals / mmd_vals.max(), "o-", label="MMD (Gaussian kernel)")
ax.axvline(a0, color="gray", linestyle="--")
ax.set_xlabel("candidate a"); ax.set_ylabel("objective (rescaled to own max)")
ax.set_title(f"Only MMD localizes the true shape parameter (true a = {a0})")
ax.legend()
plt.show()

## So have we completely solved the problem?

The MMD curve above used one particular bandwidth. What happens at other values of `h`?

In [ ]:
h_values = [0.03, 0.3, 3]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, h in zip(axes, h_values):
    vals = np.array([
        np.mean([mmd2_gaussian(x_obs, simulate_shape(n_shape, a, rng), h) for _ in range(B_rep)])
        for a in a_grid
    ])
    ax.plot(a_grid, vals, "o-", color="steelblue")
    ax.axvline(a0, color="gray", linestyle="--")
    ax.set_title(f"h = {h}")
    ax.set_xlabel("candidate a")
axes[0].set_ylabel("MMD^2")
fig.suptitle("Bandwidth controls what MMD can see")
plt.tight_layout()
plt.show()

Too small ($h=0.03$): only nearly-identical pairs register as similar at all -- the true signal is there but noisier. Good ($h=0.3$): a more convincing minimum. Too large ($h=3$): the kernel treats every pair of points as almost equally similar, which washes out the signal.

# Section 7: Indirect inference with an SIR auxiliary model

Reuse the SIR simulator from Lectures 4-5. The auxiliary model is a quadratic fit to `log(I_t + 1)` over the first 20 steps -- level, growth, curvature, in three numbers.

In [ ]:
N_pop, I0, gamma0, window = 1000, 20, 0.10, 20

def aux_fit(I_traj, window):
    te = np.arange(window + 1)
    y = np.log(np.maximum(I_traj[:window + 1], 1))
    return np.polynomial.polynomial.polyfit(te, y, 2)  # (intercept, linear, quadratic)

sim_example = simulate_sir(window, 5 * gamma0, gamma0, N_pop - I0, I0, 0, rng)
aux_fit(sim_example["I"], window)

In [ ]:
te = np.arange(window + 1)
y = np.log(np.maximum(sim_example["I"][:window + 1], 1))
coefs = aux_fit(sim_example["I"], window)
fitted = coefs[0] + coefs[1] * te + coefs[2] * te ** 2

fig, ax = plt.subplots()
ax.scatter(te, y, color="black")
ax.plot(te, fitted, color="firebrick", linewidth=2)
ax.set_xlabel("t"); ax.set_ylabel("log(I_t + 1)")
ax.set_title("Auxiliary model fit to one simulated epidemic")
plt.show()

$(\hat\alpha_0,\hat\alpha_1,\hat\alpha_2)$ would then be used as summaries for this particular simulation -- fit the auxiliary model to the one observed epidemic, average its fitted coefficients over many simulated epidemics at each candidate $(\beta,\gamma)$, and move $(\beta,\gamma)$ until that average resembles the observed fit.